# CRAFT Tutorial — Custom Residue AMBER Forcefield Toolkit

If you managed to parameterize any residue following my <a href="https://carlosramosg.com/amber-custom-residue-parameterization" target="_blank">previous tutorial</a>, CRAFT will make the process feel like a walk in the park.

This tutorial is not intended to be the way to use CRAFT — the recommendation is to install it and run it from wherever you prefer. However, for users who are just starting out in this field and for whom the parameterization process alone is already overwhelming, this notebook will work just fine: simply replace the paths to the files in the notebook and it will do the job. For those users, this can serve as a starting guide.

I'm honestly writing this for my 10-years-ago self, who struggled even with simple bash commands — I understand the struggle. The idea, though, is that later, these users will be able to move on to the natural and faster way of running CRAFT: drop in a PDB file, edit the config file, run craft-slurm on an HPC machine, go grab a coffee while CRAFT parameterizes the residue — that's the philosophy behind this package.

CRAFT automates AMBER force-field parameterization for non-standard amino acid residues using the RESP charge-fitting protocol. It takes a single-residue PDB and a short `config.yaml`, handles terminus capping, generates all Gaussian inputs, fits RESP charges, and calls the full AmberTools pipeline — creating simulation-ready `.prepin` and `.frcmod` files.

CRAFT also automates the parameterization for two bonded amino acids, wheter they are bonded side chain-side chain or side chain-backbone.
**Pipeline overview**

```
Phase 1    craft-run        cap termini, generate Gaussian + RESP inputs
Phase 2a   g16              B3LYP/6-31G* geometry optimisation
Phase 2b   craft-hf-input   extract optimised geometry, write HF input
Phase 2c   g16              HF/6-31G(d) ESP single-point
Phase 3    craft-amber      espgen → resp → antechamber → prepgen → parmchk2
```


## 0. Prerequisites

Before running this notebook make sure:

1. Clone the repo
2. CRAFT is installed in the active environment:
   Inside the **CRAFT repo root** type
   ```bash
   pip install .
   ```
3. AmberTools is installed and the `craft-check` command passes.
4. You are running the notebook from the **CRAFT repo root** (the directory containing `single_AA/`).

### Optional: 3D visualization

Molecular visualization uses **py3Dmol**, which is not required to run the pipeline but enhances the walkthrough. Install it with:

```bash
pip install py3Dmol
```

If py3Dmol is not available the visualization cells will print a skip message and the rest of the notebook will continue normally.

In [1]:
import subprocess, sys, shutil
from pathlib import Path

# Verify working directory
assert Path('single_AA/KME3.pdb').exists(), \
    'Run this notebook from the CRAFT repo root (the directory containing single_AA/)'

print('Working directory OK')

Working directory OK


In [2]:
# Check that CRAFT and AmberTools are available
result = subprocess.run(['craft-check'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

Python packages
----------------------------------------
  [ok]      numpy
  [ok]      pyyaml
  [ok]      rdkit  (full symmetry-based RESP equivalence)

Gaussian
----------------------------------------
  [MISSING] g16 or g09 -- not found in $PATH

AmberTools
----------------------------------------
  [ok]      espgen -> /Users/xv24323/miniconda3/envs/AmberTools25/bin/espgen
  [ok]      resp -> /Users/xv24323/miniconda3/envs/AmberTools25/bin/resp
  [ok]      antechamber -> /Users/xv24323/miniconda3/envs/AmberTools25/bin/antechamber
  [ok]      prepgen -> /Users/xv24323/miniconda3/envs/AmberTools25/bin/prepgen
  [ok]      parmchk2 -> /Users/xv24323/miniconda3/envs/AmberTools25/bin/parmchk2

  $AMBERHOME = /Users/xv24323/miniconda3/envs/AmberTools25
  [ok]      parm10.dat found

One or more required tools are missing. See above.




## Visualization setup 
#### Just run the next cell to create the structure visualization setup. This is not connected to CRAFT but makes the process more understandable.
#### No modification needed.

In [3]:
# -- Visualization setup -------------------------------------------------------
try:
    import py3Dmol
    HAS_PY3DMOL = True
    print('py3Dmol available — visualization enabled.')
except ImportError:
    HAS_PY3DMOL = False
    print('py3Dmol not installed.  Install with: pip install py3Dmol')
    print('Visualization cells will print a skip message and continue.')


def _pdb_with_bfactor(pdb_path, values):
    """Return PDB string with values written into the B-factor column."""
    lines_out = []
    val_iter = iter(values)
    with open(pdb_path) as f:
        for line in f:
            if line.startswith(('ATOM', 'HETATM')):
                v = next(val_iter, 0.0)
                line = f"{line[:60]}{v:6.2f}{line[66:].rstrip()}\n"
            lines_out.append(line)
    return ''.join(lines_out)


def _charge_to_hex(charge, max_val):
    """Map a charge value to a red/white/blue hex colour string."""
    t = max(min(charge / max_val, 1.0), -1.0)
    if t < 0:    # red (negative) → white (zero)
        r, g, b = 255, int(255 * (1 + t)), int(255 * (1 + t))
    else:         # white (zero) → blue (positive)
        r, g, b = int(255 * (1 - t)), int(255 * (1 - t)), 255
    return f'#{r:02x}{g:02x}{b:02x}'


def show_pdb(pdb_path, charges=None, label_charges=False, width=620, height=440,
             sphere_radius=0.3, background='0xffffff'):
    """Display a PDB file with py3Dmol."""
    if not HAS_PY3DMOL:
        print('(visualization skipped — install py3Dmol with: pip install py3Dmol)')
        return
    if charges is not None:
        pdb_str = _pdb_with_bfactor(pdb_path, charges)
    else:
        with open(pdb_path) as f:
            pdb_str = f.read()
    view = py3Dmol.view(width=width, height=height)
    view.addModel(pdb_str, 'pdb')
    view.setBackgroundColor(background)
    if charges is not None:
        lim = max(abs(c) for c in charges) or 1.0
        cs  = {'prop': 'b', 'gradient': 'rwb', 'min': -lim, 'max': lim}
        view.setStyle({}, {'stick': {'colorscheme': cs, 'radius': 0.08},
                           'sphere': {'radius': sphere_radius, 'colorscheme': cs}})
        if label_charges:
            from craft.cap import parse_pdb as _parse_pdb
            for atom, chg in zip(_parse_pdb(pdb_path), charges):
                view.addLabel(f"{atom['name']} {chg:+.3f}",
                              {'position': {'x': float(atom['x']),
                                            'y': float(atom['y']),
                                            'z': float(atom['z'])},
                               'fontSize': 10, 'fontColor': 'white',
                               'backgroundColor': 'black', 'backgroundOpacity': 0.6,
                               'showBackground': True})
    else:
        view.setStyle({}, {'stick': {'radius': 0.08}, 'sphere': {'radius': sphere_radius}})
    view.zoomTo()
    return view.show()


def show_xyz(atoms_xyz, charges=None, label_charges=False, atom_names=None,
             width=620, height=440, sphere_radius=0.3, background='0xffffff'):
    """
    Display a list of (elem, x, y, z) tuples as returned by parse_opt_log.

    Always loads as XYZ format so hydrogen atoms are never silently dropped.
    charges       : list of floats aligned with atoms_xyz — colours by charge.
    label_charges : annotate each atom with atom name (or element) + charge value.
    atom_names    : list of atom name strings aligned with atoms_xyz.
    """
    if not HAS_PY3DMOL:
        print('(visualization skipped — install py3Dmol with: pip install py3Dmol)')
        return

    lines = [str(len(atoms_xyz)), 'geometry']
    for elem, x, y, z in atoms_xyz:
        lines.append(f'{elem}  {x:.6f}  {y:.6f}  {z:.6f}')

    view = py3Dmol.view(width=width, height=height)
    view.addModel('\n'.join(lines), 'xyz')
    view.setBackgroundColor(background)

    if charges is not None:
        lim = max(abs(c) for c in charges) or 1.0
        view.setStyle({}, {'stick': {'radius': 0.08}, 'sphere': {'radius': sphere_radius}})
        for idx, ((elem, x, y, z), chg) in enumerate(zip(atoms_xyz, charges)):
            color = _charge_to_hex(chg, lim)
            view.setStyle({'index': idx},
                          {'stick': {'color': color, 'radius': 0.08},
                           'sphere': {'color': color, 'radius': sphere_radius}})
        if label_charges:
            for i, ((elem, x, y, z), chg) in enumerate(zip(atoms_xyz, charges)):
                name = atom_names[i] if atom_names is not None else elem
                view.addLabel(f"{name} {chg:+.3f}",
                              {'position': {'x': float(x), 'y': float(y), 'z': float(z)},
                               'fontSize': 10, 'fontColor': 'black',
                               'backgroundColor': 'white', 'backgroundOpacity': 0.7,
                               'showBackground': True})
    else:
        view.setStyle({}, {'stick': {'radius': 0.08}, 'sphere': {'radius': sphere_radius}})

    view.zoomTo()
    return view.show()

py3Dmol available — visualization enabled.


---
## 1. Single-residue workflow

**Example: trimethyllysine (KM3)**

This section walks you through the complete CRAFT pipeline for a single non-standard residue. I'm using trimethyllysine (KM3) — lysine with three methyl groups on the ε-amino nitrogen, giving a net charge of +1 on the capped model compound.

### 1.1 Input and configuration

CRAFT takes a single-residue PDB as input — a free amino acid, a residue cut from a crystal structure, or anything in between. The underlying protocol is RESP (Bayly et al., *J. Phys. Chem.* 1993, 97, 10269):

1. Cap the residue with ACE and/or NME groups.
2. Optimise the geometry at B3LYP/6-31G* with Gaussian.
3. Compute the electrostatic potential at HF/6-31G(d).
4. Fit RESP charges, keeping backbone and cap atoms fixed to their ff14SB values.
5. Generate the AMBER topology (`.prepin`) and missing parameters (`.frcmod`).

CRAFT automates steps 1, 4, and 5, and generates the input files for steps 2 and 3. The 5 steps can be run sequentially without human intervention if Gaussian is available. That's what `craft-slurm` does, more on this later.

#### Note on atom naming schemes: For CRAFT to understand where the cap terminus groups will be placed, you need to be aware that it will search for these atoms in the position that you want to parameterise.:


|position|required atom anmes|
|---|---|
| `middle` | N, CA, C, O |
| `cterm` | N, CA, C &nbsp;(no O — C-terminus is free) |
| `nterm` | CA, C, O &nbsp;(no N — N-terminus is free) |

CRAFT does not assume what occupies the free terminus, the three required atoms define a geometric plane that orients the cap group.
This means you can parameterize almost any chemical substituent at a terminal position, as long as those three atoms are present and correctly named.

<details>
<summary><b>Background — the RESP protocol and why backbone atoms are fixed</b></summary>
<br>

The RESP (Restrained ElectroStatic Potential) method fits atomic partial charges to reproduce a quantum-mechanical electrostatic potential computed on a grid of points around the molecule. The fitting is done with a hyperbolic restraint that penalizes large charges, preventing the fit from overfitting to points far from the nuclei.

For non-standard residues inserted into a protein, the backbone atoms (N, H, CA, HA, C, O) must carry the same charges as in the parent force field (ff14SB or ff19SB). If they were allowed to float freely, the backbone charge distribution would differ from all other residues, introducing artifacts at the peptide bond interfaces. CRAFT therefore freezes backbone charges to their ff14SB values and fits only the sidechain atoms.

The five-step pipeline in detail:

**Step 1 — terminus capping:** The residue is capped with acetyl (ACE) at the N-terminus and N-methylamide (NME) at the C-terminus. This mimics the peptide bond environment without the influence of adjacent residues.

**Step 2 — geometry optimisation (B3LYP/6-31G\*):** The capped model compound is optimised to a local minimum. Gaussian is used for this step.

**Step 3 — ESP single-point (HF/6-31G\*):** The electrostatic potential is computed at the Hartree–Fock level on a grid of points at the Connolly surface. The HF/6-31G(d) level was chosen by Bayly et al. because it systematically overestimates molecular dipole moments by ~10–20%, which empirically compensates for the condensed-phase polarization not captured by a gas-phase calculation.

**Step 4 — RESP fitting:** The `resp` program fits partial charges to the ESP grid with backbone charges fixed. Symmetry-equivalent atoms (e.g., the three methyl carbons of trimethyllysine) are constrained to carry identical charges.

**Step 5 — AMBER topology:** `antechamber` assigns atom types, `prepgen` builds the `.prepin` topology file, and `parmchk2` identifies and fills missing force-field parameters.

</details>

In [4]:
# Input PDB — raw uncapped residue
show_pdb('single_AA/KME3.pdb')

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [5]:
# #uncomment to see the starting BPD file
# with open('single_AA/KME3.pdb') as f:
#     print(f.read())

#### Configuration

All pipeline settings live in `config.yaml`. Key fields:

| Field | Description |
|---|---|
| `residue.input_pdb` | Path to the raw single-residue PDB |
| `residue.charge` | Net charge of the **capped model compound** (not just the sidechain) |
| `residue.position` | `middle` \| `cterm` \| `nterm` |
| `amber.forcefield` | `ff14SB` \| `ff19SB` |
| `gaussian_opt.route` | Gaussian route card for geometry optimisation |
| `gaussian_hf.route` | Gaussian route card for the ESP single-point |

I parameterize KM3 in the `middle` position (ACE-capped N-terminus, NME-capped C-terminus), which is the most common case for a residue embedded in a peptide chain.

<details>
<summary><b>Terminal positions — the three variants and what they mean</b></summary>
<br>

The RESP constraints depend on where in the peptide chain the residue will appear. For a residue in the interior, both ACE and NME are added and all backbone atoms are fixed. For a C-terminal or N-terminal residue, the free terminus must carry RESP-fitted charges rather than ff14SB fixed values.

| position | Capping applied | Fixed atoms (ff14SB) | Free atoms (RESP-fitted) |
|---|---|---|---|
| `middle` | ACE + NME | ACE, backbone N/H/CA/HA/C/O, NME | sidechain |
| `cterm` | ACE only | ACE, backbone N/H | sidechain + C/O/OXT |
| `nterm` | NME only | NME, backbone C/O | sidechain + N and its H atoms |

Each variant is parameterized independently into its own subdirectory (`KM3/middle/`, `KM3/cterm/`, `KM3/nterm/`). A complete parameterization requires all three.

**Important:** The free-terminus atoms must already be correctly protonated in your input PDB. For `cterm` the C-terminal carboxylate must have both oxygens (C, O, OXT). For `nterm` the amino group must carry the appropriate number of hydrogen atoms. CRAFT will warn if the terminus looks unusual but will not add or remove protons from the free terminus.

CRAFT inspects the terminus state automatically:
- **N-terminus:** counts hydrogen atoms within 1.3 Å of backbone N (0 = bare, 1 = N-H, 2 = NH2, 3 = NH3+)
- **C-terminus:** searches for a second oxygen bonded to backbone C (atom names O1, OXT, OT1, OT2, O2 within 1.6 Å)

</details>

In [6]:
# #uncomment to see the used configuration file
# with open('single_AA/config_middle.yaml') as f:
#     print(f.read())

### 1.2 Phase 1 — capping and pre-Gaussian inputs

`craft-run` reads the config, caps the termini, and writes everything the cluster will need:

1. **Caps the termini** with ACE and/or NME at ideal bond geometry.
2. **Writes the Gaussian geometry-optimisation input** (`KM3_opt.com`).
3. **Writes RESP input files** (`resp.in`, `resp.qin`) with backbone atoms fixed to ff14SB values.
4. **Writes the prepgen main-chain file** (`KM3.mc`).

All outputs land in `KM3/middle/`.

In [7]:
result = subprocess.run(
    ['craft-run', '--config', 'config_middle.yaml'],
    capture_output=True, text=True, cwd='single_AA'
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    sys.exit(result.returncode)

Step 1 -- Cap termini  [middle]
Input : KME3.pdb  (34 atoms, residue KM3)
  N-terminus : NH3+ -- zwitterionic free amino acid
  C-terminus : COO- -- has O1 (1 extra O), free amino acid C-terminus
  Position   : middle
  Action     : remove 3 H(s) on N, add amide-H, remove O1, add ACE + NME
Output: KM3/middle/KM3_capped.pdb
  ACE  resSeq=1 :  6 atoms
  KM3  resSeq=2 : 31 atoms
  NME  resSeq=3 :  6 atoms
  Total         : 43 atoms

Step 2 -- Geometry-optimisation Gaussian input
Input  : KM3/middle/KM3_capped.pdb  (43 atoms)
Output : KM3/middle/KM3_opt.com
  Charge / mult : 1 / 1
  nproc / mem   : 16 / 512MB

Steps 3-5 -- RESP and prepgen inputs
  resp.in  : KM3/middle/resp.in  (43 atoms, 25 sidechain free)
  resp.qin : KM3/middle/resp.qin  (43 charges, 25 sidechain free)
  KM3.mc          : HEAD=N CA=CA TAIL=C omit 12 cap atoms

All Phase 1 outputs written to: KM3/middle/

Next steps:
  1. Submit KM3/middle/KM3_opt.com to HPC
  2. Copy KM3_opt.log into KM3/middle/, then:
       craft-hf-

In [8]:
# Capped PDB — ACE (left) + KM3 (centre) + NME (right)
show_pdb('single_AA/KM3/middle/KM3_capped.pdb')

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [9]:
# #uncomment to inspect the capped PDB — ACE + KM3 + NME
# with open('single_AA/KM3/middle/KM3_capped.pdb') as f:
#     print(f.read())

In [10]:
# Inspect resp.qin — fixed backbone charges, free sidechain atoms (charge = 0.0)
with open('single_AA/KM3/middle/resp.qin') as f:
    print(f.read())

 -0.366200  0.112300  0.112300  0.112300  0.597200 -0.567900 -0.415700  0.271900
  0.033700  0.597200 -0.567900  0.000000  0.000000  0.000000  0.000000  0.000000
  0.000000  0.000000  0.000000  0.080800  0.000000  0.000000  0.000000  0.000000
  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000
  0.000000  0.000000  0.000000  0.000000  0.000000 -0.415700  0.271900 -0.149000
  0.097600  0.097600  0.097600



#### RESP charge constraints

CRAFT fixes the six backbone atoms to standard ff14SB/ff19SB values:

| Atom | Charge |
|------|--------|
| N    | −0.4157 |
| H    | +0.2719 |
| CA   | +0.0337 |
| HA   | +0.0823 |
| C    | +0.5973 |
| O    | −0.5679 |

The HA charge is adjusted by −0.0015 so that the six backbone atoms sum to exactly zero, preventing charge transfer artifacts when the residue is inserted into a protein. Sidechain atoms are free (shown as `0.0` in `resp.qin`).

<details>
<summary><b>How terminus detection and NERF capping work</b></summary>
<br>

**Terminus detection (`inspect_termini`)**

Before placing any cap atoms, CRAFT examines the input PDB:

N-terminus state is determined by counting hydrogen atoms within 1.3 Å of backbone N:

| H count | State | Action before ACE |
|---------|-------|-------------------|
| 0 | bare N (PDB cut, heavy atoms only) | inject amide H after ACE placed |
| 1 | N-H (peptide fragment) | keep H, use its position to anchor ACE via sp2_third |
| 2 | NH₂ (neutral free amine) | remove 2 H, inject amide H |
| 3 | NH₃⁺ (zwitterionic) | remove 3 H, inject amide H |

C-terminus state is determined by searching for a second oxygen bonded to backbone C (any atom named O1, OXT, OT1, OT2, or O2 within 1.6 Å of backbone C):

| State | Action before NME |
|-------|-------------------|
| C=O only (chain cut) | nothing to remove |
| COO⁻ (free amino acid) | remove O1/OXT |

**NERF algorithm (Natural Extension Reference Frame)**

Cap atom positions are computed from the existing backbone geometry. Given three atoms A → B → C whose positions are known, NERF places a fourth atom D such that:
- C–D bond length = `length`
- B–C–D bond angle = `angle_deg`
- A–B–C–D dihedral = `dihedral_deg`

The algorithm builds a local coordinate frame at C:
- `bc_hat` — unit vector along B→C (bond axis)
- `n` — cross(A−B, C−B) normalized (plane normal of A–B–C)
- `m` — cross(n, bc_hat) normalized (in-plane perpendicular)

Then: `D = C + length × (−cos(α) × bc_hat + sin(α) × cos(φ) × m + sin(α) × sin(φ) × n)`

Note: the sign convention for `n` shifts dihedral values by +180° relative to the standard chemistry definition. To achieve a standard dihedral of φ, the value φ − 180° is passed to NERF.

**RESP equivalence detection**

Chemically equivalent atoms must carry identical RESP charges. CRAFT detects equivalences using RDKit: it builds a molecular graph from the 3D coordinates using covalent radii to infer bonds, then runs the Morgan canonical ranking algorithm (`CanonicalRankAtoms` with `breakTies=False`) so that truly symmetric atoms receive the same rank. Atoms sharing a rank are grouped and constrained equal in `resp.in`.

If RDKit is not installed, CRAFT falls back to constraining only H atoms bonded to the same heavy atom. A warning is printed and you should review the constraint section in `resp.in` by hand for residues with equivalent heavy atoms.

</details>

### 1.3 Phase 2 — Gaussian on HPC

Phase 2 requires Gaussian 16, you can run it on a local machine or an HPC cluster. CRAFT generates the necessary input files; you submit them and copy the logs back if the job ran on an HPC cluster.

**Phase 2a — geometry optimisation (B3LYP/6-31G*)**
```bash
# On the HPC cluster:
g16 < KM3/middle/KM3_opt.com > KM3/middle/KM3_opt.log
```

**Phase 2b — build HF input from optimised geometry** on HPC
```bash
craft-hf-input KM3/middle/KM3_opt.log --config config_middle.yaml
```

**Phase 2b — build HF input from optimised geometry** where this tutorial exists

In [11]:
result = subprocess.run(
    ['craft-hf-input', 'KM3/middle/KM3_opt.log', '--config', 'config_middle.yaml'],
    capture_output=True, text=True, cwd='single_AA'
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    sys.exit(result.returncode)

Parsing optimised geometry from KM3/middle/KM3_opt.log ...
  Found 43 atoms in final Standard orientation block
Writing HF/6-31G(d) single-point input ...
Output : KM3/middle/KM3_hf.com
  Charge / mult : 1 / 1
  nproc / mem   : 16 / 512MB
  Atoms         : 43



**Phase 2c — HF/6-31G(d) single-point for ESP**
```bash
g16 < KM3/middle/KM3_hf.com > KM3/middle/KM3_hf.log
```

Alternatively, generate a SLURM script that runs the entire pipeline as a single job:
```bash
craft-slurm --config config_middle.yaml
cd KM3/middle && sbatch KM3_craft.sh
```

In [12]:
# #uncomment to see the generated Gaussian optimization input file from the capped pdb structure
# with open('single_AA/KM3/middle/KM3_opt.com') as f:
#     print(f.read())

For the rest of this notebook I use the **pre-computed Gaussian logs** included in the single_AA example, so you can follow Phase 3 without submitting jobs.

<details>
<summary><b>Route card details and freeze_backbone rationale</b></summary>
<br>

**Geometry optimisation route card (default):**
```
#P b3lyp/6-31g* opt
```
B3LYP/6-31G* is the standard level for geometry optimisation in the RESP protocol. The `opt` keyword requests a standard gradient optimisation to a local minimum.

**ESP single-point route card (default):**
```
#p hf/6-31g(d) SCF=Tight Pop=MK IOp(6/33=2)
```
- `Pop=MK` — use the Merz–Kollman scheme to compute ESP on a grid of points at the van der Waals surface
- `IOp(6/33=2)` — write the ESP data to the checkpoint file in the format that `espgen` expects
- `SCF=Tight` — tighter SCF convergence for more accurate electrostatics

**`freeze_backbone` for the two-residue workflow:**
When parameterizing two covalently bonded residues, the combined model compound has a long flexible backbone. Allowing full backbone relaxation often moves the system into non-physiological conformations. Setting `freeze_backbone: true` in the config freezes all backbone and cap atoms (marked with `F` in the Gaussian input) so that only the side chains are free to relax. This is the recommended setting for that particular case.

</details>

### 1.4 Phase 3 — AMBER parameterization

`craft-amber` runs the full AmberTools pipeline in sequence:

1. **`espgen`** — extracts the electrostatic potential grid from the Gaussian HF log into `esp.dat`.
2. **`resp`** — fits RESP charges using `resp.in`, `resp.qin`, and `esp.dat`. Fitted charges are written to `resp.chg`.
3. **`antechamber`** — reads the Gaussian HF log and fitted charges, assigns AMBER atom types, writes a `.ac` file. CRAFT then remaps atom names in the `.ac` file to match the capped PDB.
4. **`prepgen`** — reads the `.ac` file and the `.mc` file from Phase 1, assembles the `.prepin` topology.
5. **`parmchk2`** — scans the `.prepin` for missing parameters and writes `_gaff.frcmod` and `_ff14SB.frcmod`.

In [13]:
result = subprocess.run(
    ['craft-amber', 'KM3/middle/KM3_hf.log', '--config', 'config_middle.yaml'],
    capture_output=True, text=True, cwd='single_AA'
)

## A non Black Box approach

To understand what CRAFT is doing, it is important for you to check the output of the Craft-Amber module. 

This will show you the order in which the commands are executed, as well as the files used at each stage.

In [14]:
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    sys.exit(result.returncode)

Residue  : KM3  (charge +1)
Position : middle
Log      : KM3/middle/KM3_hf.log
MC file  : /Users/xv24323/Documents/github/CRAFT/single_AA/KM3/middle/KM3.mc


-- espgen ------------------------------------------------------------
  $ espgen -i /Users/xv24323/Documents/github/CRAFT/single_AA/KM3/middle/KM3_hf.log -o esp.dat


-- resp --------------------------------------------------------------
  $ resp -O -i resp.in -o resp.out -p resp.pch -t resp.chg -q resp.qin -e esp.dat

-- antechamber -------------------------------------------------------
  $ antechamber -fi gout -i /Users/xv24323/Documents/github/CRAFT/single_AA/KM3/middle/KM3_hf.log -bk KM3 -fo ac -o KM3.ac -c rc -cf resp.chg -at amber -nc 1
Info: acdoctor mode is on: check and diagnose problems in the input file.
Info: The atom type is set to amber; the options available to the -at flag are
      gaff, gaff2, amber, bcc, abcg2, and sybyl.

-- Check Format for Gaussian Output File --
   Status: pass

-- remap atom names -------

# Results visualization

If you got here, the para meterization process is done. So Let's take a look first ot the assigned partial charges on the starting PDB structure 

### Optimised geometry coloured by RESP-fitted charges
#### red = negative  /  white = zero  /  blue = positive

In [15]:
# Show RESP-fitted charges alongside atom names from the capped PDB
from craft.cap import parse_pdb
from craft.gaussian import parse_opt_log

atom_names = [a['name'] for a in parse_pdb('single_AA/KM3/middle/KM3_capped.pdb')]
opt_geom = parse_opt_log('single_AA/KM3/middle/KM3_opt.log')
charges = [float(x) for x in open('single_AA/KM3/middle/resp.chg').read().split()]
show_xyz(opt_geom, label_charges=True, charges=charges, atom_names=atom_names, width=1000, height=650)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [16]:
# atoms   = parse_pdb('single_AA/KM3/middle/KM3_capped.pdb')
# print(f"{'Atom':<6} {'Res':<6} {'Charge':>8}")
# print('-' * 24)
# for atom, charge in zip(atoms, charges):
#     print(f"{atom['name']:<6} {atom['resName']:<6} {charge:>8.4f}")

<details>
<summary><b>DU atom type resolution — what happens when antechamber produces unknown types</b></summary>
<br>

On rare occasions `antechamber` assigns the placeholder atom type `DU` to atoms it cannot classify. This typically happens for unusual bonding environments.

CRAFT detects `DU` assignments automatically and attempts to resolve them in two ways:

1. **GAFF2 probe:** CRAFT re-runs `antechamber` with `-at gaff2` on the same system and transfers the GAFF2 atom type to the atom that was assigned `DU`. This works for most common functional groups.

2. **`atom_type_overrides`:** If the GAFF2 probe also fails, you can specify the correct atom type manually in `config.yaml`:
   ```yaml
   amber:
     atom_type: amber
     atom_type_overrides:
       N3: c3   # force atom named N3 to type c3
   ```

When `DU` types are resolved, CRAFT prints a warning showing which atoms were affected and what types were assigned. You should review the `parmchk2` output (`_gaff.frcmod`) to confirm that parameters exist for the new atom types.

</details>

### 1.5 Outputs and tLeap usage

All outputs for a given variant land in `KM3/middle/`:

| File | Description |
|---|---|
| `KM3.prepin` | Residue topology for tLeap |
| `KM3_gaff.frcmod` | Missing GAFF parameters |
| `KM3_ff14SB.frcmod` | Missing ff14SB parameters |
| `KM3_capped.pdb` | Capped input structure |
| `KM3_opt.com` / `.log` | Gaussian geometry optimisation |
| `KM3_hf.com` / `.log` | Gaussian HF/ESP single-point |
| `resp.in`, `resp.qin` | RESP input files |
| `KM3.mc` | Prepgen main-chain file |

In [17]:
output_files_middle = [
    'single_AA/KM3/middle/KM3.prepin',
    'single_AA/KM3/middle/KM3_gaff.frcmod',
    'single_AA/KM3/middle/KM3_ff14SB.frcmod',
    'single_AA/KM3/middle/KM3_capped.pdb',
    'single_AA/KM3/middle/KM3_opt.com',
    'single_AA/KM3/middle/KM3_hf.com',
    'single_AA/KM3/middle/resp.in',
    'single_AA/KM3/middle/resp.qin',
    'single_AA/KM3/middle/KM3.mc',
]

print(f"{'File':<50} {'Status'}")
print('-' * 58)
for f in output_files_middle:
    status = 'OK' if Path(f).exists() else 'MISSING'
    print(f"{f:<50} {status}")

File                                               Status
----------------------------------------------------------
single_AA/KM3/middle/KM3.prepin                    OK
single_AA/KM3/middle/KM3_gaff.frcmod               OK
single_AA/KM3/middle/KM3_ff14SB.frcmod             OK
single_AA/KM3/middle/KM3_capped.pdb                OK
single_AA/KM3/middle/KM3_opt.com                   OK
single_AA/KM3/middle/KM3_hf.com                    OK
single_AA/KM3/middle/resp.in                       OK
single_AA/KM3/middle/resp.qin                      OK
single_AA/KM3/middle/KM3.mc                        OK


#### Using the outputs in tLeap

The `.prepin` and `.frcmod` files are loaded into tLeap alongside the standard force field:

```
# tleap input example
source leaprc.protein.ff14SB

loadAmberPrep   KM3/middle/KM3.prepin
loadAmberParams KM3/middle/KM3_gaff.frcmod
loadAmberParams KM3/middle/KM3_ff14SB.frcmod

mol = sequence { ACE KM3 NME }
saveAmberParm mol system.parm7 system.rst7
quit
```

For a full protein containing this residue, load the PDB after the `loadAmberPrep` and `loadAmberParams` calls. The residue name in the PDB must match the three-letter code (`KM3` in this example).

#### All three terminal positions

The terminal steps are only required if you are aiming to use the reside as C or N terminal. Here the single_AA folder includes configs for all three: `config_middle.yaml`, `config_cterm.yaml`, and `config_nterm.yaml`.

In [18]:
positions = [
    ('middle', 'KM3/middle/KM3_hf.log',      'config_middle.yaml'),
    ('cterm',  'KM3/cterm/KM3_cterm_hf.log', 'config_cterm.yaml'),
    ('nterm',  'KM3/nterm/KM3_nterm_hf.log', 'config_nterm.yaml'),
]

for position, hf_log, config in positions:
    sep = '=' * 60
    print(f'\n{sep}')
    print(f'Position: {position}')
    print(sep)
    result = subprocess.run(
        ['craft-amber', hf_log, '--config', config],
        capture_output=True, text=True, cwd='single_AA'
    )
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr)


Position: middle
Residue  : KM3  (charge +1)
Position : middle
Log      : KM3/middle/KM3_hf.log
MC file  : /Users/xv24323/Documents/github/CRAFT/single_AA/KM3/middle/KM3.mc


-- espgen ------------------------------------------------------------
  $ espgen -i /Users/xv24323/Documents/github/CRAFT/single_AA/KM3/middle/KM3_hf.log -o esp.dat


-- resp --------------------------------------------------------------
  $ resp -O -i resp.in -o resp.out -p resp.pch -t resp.chg -q resp.qin -e esp.dat

-- antechamber -------------------------------------------------------
  $ antechamber -fi gout -i /Users/xv24323/Documents/github/CRAFT/single_AA/KM3/middle/KM3_hf.log -bk KM3 -fo ac -o KM3.ac -c rc -cf resp.chg -at amber -nc 1
Info: acdoctor mode is on: check and diagnose problems in the input file.
Info: The atom type is set to amber; the options available to the -at flag are
      gaff, gaff2, amber, bcc, abcg2, and sybyl.

-- Check Format for Gaussian Output File --
   Status: pass

-- remap 

In [19]:
# Summary of output files for all three positions
output_files = [
    'single_AA/KM3/middle/KM3.prepin',
    'single_AA/KM3/middle/KM3_ff14SB.frcmod',
    'single_AA/KM3/cterm/KM3_cterm.prepin',
    'single_AA/KM3/cterm/KM3_cterm_ff14SB.frcmod',
    'single_AA/KM3/nterm/KM3_nterm.prepin',
    'single_AA/KM3/nterm/KM3_nterm_ff14SB.frcmod',
]

print(f"{'File':<55} {'Status'}")
print('-' * 62)
for f in output_files:
    status = 'OK' if Path(f).exists() else 'MISSING'
    print(f"{f:<55} {status}")

File                                                    Status
--------------------------------------------------------------
single_AA/KM3/middle/KM3.prepin                         OK
single_AA/KM3/middle/KM3_ff14SB.frcmod                  OK
single_AA/KM3/cterm/KM3_cterm.prepin                    OK
single_AA/KM3/cterm/KM3_cterm_ff14SB.frcmod             OK
single_AA/KM3/nterm/KM3_nterm.prepin                    OK
single_AA/KM3/nterm/KM3_nterm_ff14SB.frcmod             OK


---
## 2. Two-residue covalent parameterization

**Example: CYA–ASA crosslink**

**Double PDB Approach**

Some non-standard residues exist only as part of a **covalent crosslink** between two residues — disulfide bonds (Cys–Cys), isopeptide bonds (Lys–Glu), or thioether/thioester linkages. CRAFT's bond workflow parameterizes both residues simultaneously so that the shared bond interface receives correct charges and atom types for its bonded state.

This section uses the **CYA–ASA** example: a cysteine derivative (CYA) whose sulfur (SG) forms a covalent bond with a carbon (CG) of an aspartate derivative (ASA).

### 2.1 Introduction and configuration

For two covalently bonded residues, CRAFT builds a combined model compound containing both residues capped independently: `ACE–CYA–NME + ACE–ASA–NME` (6 residue groups total). A joint RESP fit is run on the entire six-residue system so that the atoms at the bond interface receive charges that reflect the bonded electronic environment.

The config file uses `config_bond.yaml` and the `bond:` key to specify the inter-residue covalent bond:

<details>
<summary><b>How the combined model compound is assembled</b></summary>
<br>

When `bond.combined_pdb` is not provided, CRAFT assembles the combined model compound in three steps:

1. **Cap each residue independently** using the same NERF algorithm as the single-residue workflow. This gives ACE–CYA–NME and ACE–ASA–NME as separate systems.

2. **Assemble the bond geometry:** The reactive atoms (`SG` on CYA, `CG` on ASA) are brought to the specified bond length (default 1.5 Å). An angle correction is applied to ensure that the bond geometry is chemically reasonable.

3. **UFF minimization:** A short UFF (Universal Force Field) minimization relaxes the side chains in the vicinity of the new bond while keeping the backbone frozen. This removes steric clashes introduced by the assembly step.

Alternatively, if you already have a combined PDB (e.g., from a crystal structure), set `bond.combined_pdb` to its path. CRAFT will use the existing geometry and skip the assembly step, proceeding directly to Gaussian input generation. Atom name conflicts between the two residue blocks are resolved automatically.

The `resSeq` numbers in the combined PDB follow the convention: 1=ACE, 2=residue1, 3=NME, 4=ACE, 5=residue2, 6=NME. A `rename_map.json` file records any atom renames applied to avoid name collisions.

</details>

In [20]:
with open('two_bonded_AA/config_bond.yaml') as f:
    print(f.read())

# CRAFT bond configuration reference
#
# Copy this file, fill in the values for your two residues, and save it as
# config.yaml in your working directory.  Use this template when parameterizing
# two residues that form a covalent bond through their side chains — disulfide
# bonds, isopeptide bonds, NOS bonds (Lys–Cys), acylation of Ser/Cys, etc.
#
# For single-residue parameterization use config.yaml instead.
#
#
# -- Commands ------------------------------------------------------------------
#
#   Phase 1 (local):
#     craft-run --config config.yaml
#
#   Phase 2b (local, after opt log arrives from HPC):
#     craft-hf-input <r1>_<r2>/<r1>_<r2>_bond_opt.log \
#                    --charge <total> --config config.yaml
#
#   Phase 3 (local, after HF log arrives from HPC):
#     craft-amber <r1>_<r2>/<r1>_<r2>_bond_hf.log --config config.yaml
#
#   Single SLURM job (full pipeline):
#     craft-slurm --config config.yaml
#     cd <r1>_<r2> && sbatch <r1>_<r2>_bond_craft.sh
#
# -- Output 

### 2.2 Input structures

In [21]:
# Residue 1 — CYA
show_pdb('two_bonded_AA/CYA.pdb')

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [22]:
# Residue 2 — ASA
show_pdb('two_bonded_AA/ASA.pdb')

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### 2.3 Phase 1 — combined model compound

`craft-run` detects the bond config, caps each residue independently with ACE/NME, assembles the combined model compound (6 residue groups: ACE–CYA–NME + ACE–ASA–NME), and writes:

- `CYA_ASA_bond_combined.pdb` — the six-residue model compound
- `CYA_ASA_bond_opt.com` — frozen-backbone Gaussian optimisation input
- `resp.in` / `resp.qin` — RESP control files with backbone atoms fixed
- `CYA.mc` / `ASA.mc` — prepgen main-chain files for each residue
- `rename_map.json` — atom rename records for the second residue block

In [23]:
result = subprocess.run(
    ['craft-run', '--config', 'config_bond.yaml'],
    capture_output=True, text=True, cwd='two_bonded_AA'
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

Step 1 -- Cap CYA  [middle]
Input : CYA.pdb  (11 atoms, residue CYA)
  N-terminus : N-H -- single amide H (cut from PDB with H, or peptide fragment)
  C-terminus : COO- -- has OXT (1 extra O), free amino acid C-terminus
  Position   : middle
  Action     : keep existing amide-H, use it to anchor ACE, remove OXT, add ACE + NME
Output: CYA_ASA/CYA/CYA_capped.pdb
  ACE  resSeq=1 :  6 atoms
  CYA  resSeq=2 : 10 atoms
  NME  resSeq=3 :  6 atoms
  Total         : 22 atoms

Step 2 -- Cap ASA  [middle]
Input : ASA.pdb  (12 atoms, residue ASA)
  N-terminus : N-H -- single amide H (cut from PDB with H, or peptide fragment)
  C-terminus : COO- -- has OXT (1 extra O), free amino acid C-terminus
  Position   : middle
  Action     : keep existing amide-H, use it to anchor ACE, remove OXT, add ACE + NME
Output: CYA_ASA/ASA/ASA_capped.pdb
  ACE  resSeq=1 :  6 atoms
  ASA  resSeq=2 : 11 atoms
  NME  resSeq=3 :  6 atoms
  Total         : 23 atoms

Step 3 -- Assemble combined model  (SG—CG, None Å)
Combi

In [24]:
# Combined model compound: ACE–CYA–NME (residues 1–3) + ACE–ASA–NME (residues 4–6)
show_pdb('two_bonded_AA/CYA_ASA/CYA_ASA_bond_combined.pdb')

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### Note: Single PDB vs double PDB approach

While CRAFT does its best to position the two residues, this could result in geometries that differ greatly from those in your real system. For the case of two bonded amino acids, I recommend using a pre-capped PDB containing the two amino acids in a single PDB file from your actual project. This is where coding ends and scientific criteria begin.

If a single PDB file containing the expected sequence of residues is provided in the `combined_pdb` field in the `bond` section of the `config_bond.yaml` file, CRAFT will follow this path automatically.

This cell provides a detailed guide on how to write those residues in the single PDB approach. While it may appear somewhat daunting at first, it is actually quite straightforward if you are familiar with working with full PDB protein systems.

<details>
<summary><b>Amino Acids Sequence Structure for the Single PDB Approach</b></summary>
<br>

The combined model compound is built by capping each residue independently and concatenating them in a single PDB file. The `resSeq` numbering CRAFT expects for two residues in `middle` position is:

| resSeq | Group | Role |
|--------|-------|------|
| 1 | ACE | N-cap of residue 1 |
| 2 | RES1 | first residue — side-chain bond to RES2 |
| 3 | NME | C-cap of residue 1 |
| 4 | ACE | N-cap of residue 2 |
| 5 | RES2 | second residue — side-chain bond to RES1 |
| 6 | NME | C-cap of residue 2 |

When either residue sits at a chain terminus, the corresponding cap is omitted:

| RES1 `position` \ RES2 `position` | `middle` | `cterm` | `nterm` |
|-----------------------------------|----------|---------|----------|
| **`middle`** | ACE–RES1–NME / ACE–RES2–NME | ACE–RES1–NME / ACE–RES2 | ACE–RES1–NME / RES2–NME |
| **`cterm`** | ACE–RES1 / ACE–RES2–NME | ACE–RES1 / ACE–RES2 | ACE–RES1 / RES2–NME |
| **`nterm`** | RES1–NME / ACE–RES2–NME | RES1–NME / ACE–RES2 | RES1–NME / RES2–NME |

The `/` separates the two capped fragments. The bond between RES1 and RES2 is always present COMPLETE THIS TOMORROW regardless of position.

</details>

In [25]:
# Gaussian frozen-backbone optimisation input
# Backbone and cap atoms are frozen (F); only side chains are free to move
with open('two_bonded_AA/CYA_ASA/CYA_ASA_bond_opt.com') as f:
    content = f.read()
lines = content.splitlines()
print('\n'.join(lines[:60]))
if len(lines) > 60:
    print(f'... ({len(lines) - 60} more lines)')

%nprocshared=16
%mem=512MB
#P b3lyp/6-31g* opt(modredundant)

CYA_ASA_bond_combined  covalent bond complex backbone-frozen optimisation

0 1
 C       108.97800000     90.91200000     76.22900000
 H       109.79700000     91.28400000     75.61300000
 H       109.04400000     89.82700000     76.30500000
 H       109.04400000     91.35000000     77.22500000
 C       107.65000000     91.29600000     75.59400000
 O       106.60800000     91.16500000     76.23200000
 N       107.62500000     91.77300000     74.34700000
 H       108.48500000     91.88500000     73.81000000
 C       106.36000000     92.15200000     73.70700000
 H       106.03500000     93.11600000     74.09700000
 C       106.51200000     92.26700000     72.14600000
 H       107.41300000     91.74200000     71.82900000
 H       105.64300000     91.82100000     71.66200000
 S       106.73900000     94.07300000     71.65300000
 C       105.32700000     91.05300000     73.94500000
 O       105.63100000     89.89100000     74.0810

### 2.4 Phase 2 — Gaussian on HPC

The same three-step Gaussian workflow applies, but the system is larger (two capped residues instead of one).

```bash
# Phase 2a — geometry optimisation (backbone frozen)
g16 < CYA_ASA/CYA_ASA_bond_opt.com > CYA_ASA/CYA_ASA_bond_opt.log

# Phase 2b — extract optimised geometry, write HF input
craft-hf-input CYA_ASA/CYA_ASA_bond_opt.log --charge 0 --config config_bond.yaml

# Phase 2c — HF/ESP single-point
g16 < CYA_ASA/CYA_ASA_bond_hf.com > CYA_ASA/CYA_ASA_bond_hf.log
```

Or as a single SLURM job:
```bash
craft-slurm --config config_bond.yaml
cd CYA_ASA && sbatch CYA_ASA_bond_craft.sh
```

In [26]:
opt_geom = parse_opt_log('two_bonded_AA/CYA_ASA/CYA_ASA_bond_opt.log')
show_xyz(opt_geom)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### 2.5 Phase 3 — AMBER parameterization

`craft-amber` detects the bond config and runs the combined pipeline:
`espgen → resp → antechamber (combined .ac) → prepgen × 2 → parmchk2`

`antechamber` runs on the combined HF log so the reactive atoms at the bond interface receive correct AMBER types for their bonded state. `prepgen` is run twice with per-residue OMIT lists to extract separate `.prepin` files. A single shared `.frcmod` covers missing parameters for both residues.

In [27]:
result = subprocess.run(
    ['craft-amber', 'CYA_ASA/CYA_ASA_bond_hf.log', '--config', 'config_bond.yaml'],
    capture_output=True, text=True, cwd='two_bonded_AA'
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

Residues     : CYA (charge +0) + ASA (charge +0)
Total charge : +0
Log          : CYA_ASA/CYA_ASA_bond_hf.log
Rename map   : 21 atom(s) will be restored in ASA.prepin


-- espgen ------------------------------------------------------------
  $ espgen -i /Users/xv24323/Documents/github/CRAFT/two_bonded_AA/CYA_ASA/CYA_ASA_bond_hf.log -o esp.dat


-- resp --------------------------------------------------------------
  $ resp -O -i resp.in -o resp.out -p resp.pch -t resp.chg -q resp.qin -e esp.dat

-- antechamber -------------------------------------------------------
  $ antechamber -fi gout -i /Users/xv24323/Documents/github/CRAFT/two_bonded_AA/CYA_ASA/CYA_ASA_bond_hf.log -bk CYA -fo ac -o CYA_ASA_bond.ac -c rc -cf resp.chg -at amber -nc 0
Info: acdoctor mode is on: check and diagnose problems in the input file.
Info: The atom type is set to amber; the options available to the -at flag are
      gaff, gaff2, amber, bcc, abcg2, and sybyl.

-- Check Format for Gaussian Output File --
   S

In [28]:
from craft.cap import parse_pdb

combined_pdb = 'two_bonded_AA/CYA_ASA/CYA_ASA_bond_combined.pdb'
atoms   = parse_pdb(combined_pdb)
charges = [float(x) for x in open('two_bonded_AA/CYA_ASA/resp.chg').read().split()]

print(f"{'Atom':<8} {'Res':<6} {'Seg':>4}  {'Charge':>8}")
print('-' * 32)
for atom, charge in zip(atoms, charges):
    seg = 'CYA' if atom['resSeq'] == 2 else ('ASA' if atom['resSeq'] == 5 else 'cap')
    print(f"{atom['name']:<8} {atom['resName']:<6} {seg:>4}  {charge:>8.4f}")

Atom     Res     Seg    Charge
--------------------------------
CH3      ACE     cap   -0.3662
H1       ACE     cap    0.1123
H2       ACE     cap    0.1123
H3       ACE     cap    0.1123
C1       ACE     cap    0.5972
O1       ACE     cap   -0.5679
N        CYA     CYA   -0.4157
H        CYA     CYA    0.2719
CA       CYA     CYA    0.0337
HA       CYA     CYA    0.0808
CB       CYA     CYA    0.1635
HB2      CYA     CYA    0.0165
HB3      CYA     CYA    0.0165
SG       CYA     CYA   -0.2754
C        CYA     CYA    0.5972
O        CYA     CYA   -0.5679
N1       NME     cap   -0.4157
H4       NME     cap    0.2719
C2       NME     cap   -0.1490
H11      NME     cap    0.0976
H21      NME     cap    0.0976
H31      NME     cap    0.0976
CH32     ACE     cap   -0.3662
H12      ACE     cap    0.1123
H22      ACE     cap    0.1123
H32      ACE     cap    0.1123
C12      ACE     cap    0.5972
O12      ACE     cap   -0.5679
N2       ASA     ASA   -0.4157
H5       ASA     ASA    0.2719
CA2   

In [29]:
# Show RESP-fitted charges alongside atom names from the capped PDB
from craft.cap import parse_pdb
from craft.gaussian import parse_opt_log

atom_names = [a['name'] for a in parse_pdb('two_bonded_AA/CYA_ASA/CYA_ASA_bond_combined.pdb')]
opt_geom = parse_opt_log('two_bonded_AA/CYA_ASA/CYA_ASA_bond_opt.log')
charges = [float(x) for x in open('two_bonded_AA/CYA_ASA/resp.chg').read().split()]
show_xyz(opt_geom, label_charges=True, charges=charges, atom_names=atom_names, width=1000, height=650)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [30]:
# Summary of output files — two prepins + shared frcmod
output_files = [
    'two_bonded_AA/CYA_ASA/CYA_ASA_bond.ac',
    'two_bonded_AA/CYA_ASA/CYA/CYA.prepin',
    'two_bonded_AA/CYA_ASA/ASA/ASA.prepin',
    'two_bonded_AA/CYA_ASA/CYA_ASA_bond_gaff.frcmod',
    'two_bonded_AA/CYA_ASA/CYA_ASA_bond_ff14SB.frcmod',
]

print(f"{'File':<58} {'Status'}")
print('-' * 65)
for f in output_files:
    status = 'OK' if Path(f).exists() else 'MISSING'
    print(f"{f:<58} {status}")

File                                                       Status
-----------------------------------------------------------------
two_bonded_AA/CYA_ASA/CYA_ASA_bond.ac                      OK
two_bonded_AA/CYA_ASA/CYA/CYA.prepin                       OK
two_bonded_AA/CYA_ASA/ASA/ASA.prepin                       OK
two_bonded_AA/CYA_ASA/CYA_ASA_bond_gaff.frcmod             OK
two_bonded_AA/CYA_ASA/CYA_ASA_bond_ff14SB.frcmod           OK


### 2.6 tLeap usage for bonded residues

Both residues share a single `.frcmod`. Load both `.prepin` files and use a `bond` command to define the crosslink:

```
source leaprc.protein.ff14SB

loadAmberPrep  CYA_ASA/CYA/CYA.prepin
loadAmberPrep  CYA_ASA/ASA/ASA.prepin
loadAmberParams CYA_ASA/CYA_ASA_bond_ff14SB.frcmod

mol = loadPdb protein.pdb
bond mol.42.SG mol.67.CG    # residue numbers from your PDB

saveAmberParm mol system.prmtop system.inpcrd
quit
```

The `bond` command creates the covalent crosslink in the AMBER topology. Both residues must appear in the PDB with their correct three-letter names (CYA and ASA).

---
## 3. Notes and edge cases

### Charged sidechains

Set `charge` in `config.yaml` to the net charge of the **entire capped molecule**, not just the sidechain. For a cationic residue like trimethyllysine (KM3) the capped molecule carries `charge: +1` even though the cap groups are neutral. For the two-residue workflow, `residue1.charge` and `residue2.charge` are the charges of each individual residue in its bonded state; the total QM charge is their sum.

### Atom name conventions

CRAFT uses the atom names exactly as they appear in the input PDB. If your PDB uses non-standard names, they will propagate through to the `.prepin`. Atom names must be unique within the residue — CRAFT validates this before running.

### Symmetry equivalences without RDKit

Without RDKit installed, CRAFT constrains only H atoms bonded to the same heavy atom. For residues with chemically equivalent heavy atoms (methyl groups, aromatic rings, symmetric substituents), this is insufficient. Either install RDKit or edit the `resp.in` intra-molecular constraint section by hand before running `craft-amber`.

```bash
pip install rdkit  # or: conda install -c conda-forge rdkit
```

### Checking RESP charges

Always verify that the fitted RESP charges make chemical sense. The RESP output is printed to stdout by `craft-amber` and written into the `.prepin` file. Compare chemically equivalent atoms — they should carry identical charges if CRAFT applied the symmetry constraints correctly. For example, the three methyl carbons of trimethyllysine should each carry the same charge, and so should the nine methyl hydrogens within each group.

### Running all three positions

Change `position` in `config.yaml` and re-run each phase:

```bash
# in config.yaml: position: cterm
craft-run --config config.yaml
craft-hf-input single_AA/cterm/KM3_cterm_opt.log --config config.yaml
craft-amber    single_AA/cterm/KM3_cterm_hf.log  --config config.yaml
```

The three variants do not share intermediate files, so they can be parameterized in any order or in parallel on separate cluster nodes.

---
## 4. Developer reference

The following sections document the internal algorithms in `craft/bond.py` (two-residue assembly) and `craft/cap.py` (terminus detection and cap placement). They are intended for contributors and users who need to understand or modify the capping geometry.

<details>
<summary><b>4a. Terminus detection algorithm</b></summary>
<br>

Before placing any cap atoms, `inspect_termini()` examines the input PDB to determine the actual state of both termini. This drives which atoms must be removed and what gets reported to the user.

**N-terminus states**

The N-terminus is characterized by counting hydrogen atoms within 1.3 Å of the backbone N (the maximum N–H bond length, well below any non-bonded distance):

| H count within 1.3 Å of N | State | Action before ACE |
|---|---|---|
| 0 | Bare N — PDB cut, no H | Remove nothing; inject amide H after ACE placed |
| 1 | N-H — peptide fragment | Keep H; use its position to anchor ACE via sp2_third |
| 2 | NH₂ — neutral free amine | Remove 2 H; inject amide H |
| 3 | NH₃⁺ — zwitterionic | Remove 3 H; inject amide H |

**C-terminus states**

The C-terminus is characterized by searching for a second oxygen bonded to the backbone C (any atom named O1, OXT, OT1, OT2, or O2 within 1.6 Å of backbone C):

| State | What is present | Action |
|---|---|---|
| C=O only | Backbone C + O (carbonyl) only | Nothing to remove; NME_N placed from C, CA, N geometry |
| COO⁻ | Backbone C + O (carbonyl) + O1/OXT (carboxylate) | Remove O1/OXT; NME_N placed identically |

**Threshold rationale:**
- 1.3 Å for H: typical N–H bond is ~1.01 Å; 1.3 Å safely excludes all non-bonded H···N contacts (~2.7 Å minimum)
- 1.6 Å for O: captures C–O bonds (1.23–1.36 Å) while excluding non-bonded C···O contacts (~2.8 Å minimum)

</details>

<details>
<summary><b>4b. NERF algorithm — the math</b></summary>
<br>

**The problem**

Given three atoms A → B → C whose positions are already known, place a fourth atom D such that:
- The C–D bond length equals `length`
- The B–C–D bond angle equals `angle_deg`
- The A–B–C–D dihedral (torsion) equals `dihedral_deg`

This is exactly what is needed to extend a backbone one atom at a time.

**Building the local coordinate frame**

Three orthogonal basis vectors are constructed at C:

1. **Bond axis** — `bc_hat` points along B→C:
   ```
   bc = C − B
   bc_hat = bc / |bc|
   ```

2. **Plane normal** — `n` is perpendicular to the plane containing A, B, C:
   ```
   n = cross(A−B, bc)    # note: sign flipped vs standard chemistry convention
   if |n| < 1e-10:       # A,B,C collinear — fallback
       n = cross(bc_hat, arbitrary_perp)
   n /= |n|
   ```

   The two conventions point in opposite directions from the A–B–C plane:
   ```
        n₁  ↑  (standard: cross(B−A, bc))
             |
   A────────B────C      ← A–B–C plane
             |
        n   ↓  (CRAFT:    cross(A−B, bc))
   ```
   Because n = −n₁, every dihedral angle CRAFT computes is shifted by +180° relative
   to the standard chemistry definition.

3. **In-plane perpendicular** — `m` lies in the A–B–C plane, perpendicular to the bond axis:
   ```
   m = cross(n, bc_hat)
   ```

**Placing D**

D is parameterized in spherical coordinates within the local frame:

```
D = C + length × (
    −cos(α) × bc_hat       # axial component
  + sin(α) × cos(φ) × m   # in-plane sweep
  + sin(α) × sin(φ) × n   # out-of-plane sweep
)
```

where α = `angle_deg` and φ = `dihedral_deg` (the values passed to the function).

**Sign convention — the critical detail**

`n` is computed as `cross(A−B, bc)`, whereas the standard chemistry convention uses `cross(B−A, bc)`. These differ by a sign, which shifts dihedral values by +180° relative to the standard definition.

Consequence: to achieve a standard dihedral of φ, pass φ − 180° to `nerf()`.

| Placement | Standard dihedral wanted | Value passed to nerf() | Why this standard value |
|---|---|---|---|
| ACE_C | C–CA–N–ACE_C = 180° (trans ω) | 0° | Extended trans peptide bond |
| ACE_O | CA–N–ACE_C–O = 0° (cis to CA) | 180° | Carbonyl O cis to CA in trans amide |
| Amide H on N | CA–ACE_C–N–H = 180° | 0° | H trans to CA across ACE_C–N |
| NME_N | N–CA–C–NME_N = 180° (extended ψ) | 0° | Extended backbone conformation |
| NME_H | CA–C–NME_N–H = 0° (cis to CA) | 180° | H cis to CA in trans amide |

</details>

<details>
<summary><b>4c. Cap atom placements — full tables</b></summary>
<br>

**ACE group**

| Atom | Function | A, B, C reference atoms | Length (Å) | Angle (°) | Dihedral passed |
|---|---|---|---|---|---|
| ACE_C | nerf | C_res, CA_res, N_res | 1.335 | 121.0 | 0° (std 180°, trans ω) |
| ACE_O | nerf | CA_res, N_res, ACE_C | 1.229 | 120.5 | 180° (std 0°, O cis to CA) |
| ACE_CH3 | sp2_third | center=ACE_C, p1=ACE_O, p2=N_res | 1.522 | — | — |
| H1, H2, H3 | methyl_H | C_pos=ACE_CH3, bonded_to=ACE_C | 1.090 | 109.47 | 0°/120°/240° |
| H (amide) | nerf | CA_res, ACE_C, N_res | 1.010 | 118.0 | 0° (std 180°) |

**NME group**

| Atom | Function | A, B, C reference atoms | Length (Å) | Angle (°) | Dihedral passed |
|---|---|---|---|---|---|
| NME_N | nerf | N_res, CA_res, C_res | 1.335 | 116.6 | 0° (std 180°, extended ψ) |
| NME_H | nerf | CA_res, C_res, NME_N | 1.010 | 119.0 | 180° (std 0°, H cis to CA) |
| NME_CH3 | sp2_third | center=NME_N, p1=C_res, p2=NME_H | 1.449 | — | — |
| H1, H2, H3 | methyl_H | C_pos=NME_CH3, bonded_to=NME_N | 1.090 | 109.47 | 0°/120°/240° |

**Supporting functions**

`sp2_third(center, p1, p2, bond_length)` — places the third bond of an sp2 center. For a flat sp2 atom, the three bond unit vectors sum to zero. The third bond direction is therefore exactly opposite to the resultant of the first two:

```python
u1 = p1 - center;  u1 /= np.linalg.norm(u1)
u2 = p2 - center;  u2 /= np.linalg.norm(u2)
v = -(u1 + u2)
return center + bond_length * v / np.linalg.norm(v)
```

`methyl_H(C_pos, bonded_to, bond_length)` — places three tetrahedral H positions on –CH₃. Each H sits at 109.471° from the C→bonded_to axis, spaced 120° apart rotationally:

```python
# Position formula for each H (t = 0°, 120°, 240°):
H = C_pos + bond_length × (
    cos(109.47°) × axis
  + sin(109.47°) × cos(t) × perp
  + sin(109.47°) × sin(t) × perp2
)
```

</details>

<details>
<summary><b>4d. Geometry parameters reference</b></summary>
<br>

All hardcoded values and their source. Change these if you want a different starting conformation or if a force field uses different ideal geometry.

| Parameter | Value | Source / rationale |
|---|---|---|
| C–N peptide bond (ACE–RES, RES–NME) | 1.335 Å | AMBER ff14SB ideal peptide C–N |
| C=O carbonyl (ACE) | 1.229 Å | Ideal sp2 C=O |
| C–C (ACE_C–CH3) | 1.522 Å | Ideal sp3–sp2 C–C |
| N–H amide | 1.010 Å | AMBER ff14SB ideal N–H |
| N–C (NME_N–CH3) | 1.449 Å | Ideal sp2–sp3 N–C |
| C–H methyl | 1.090 Å | Ideal sp3 C–H |
| CA–N–C angle (at N) | 121.0° | Standard peptide bond angle |
| N–C=O angle (ACE) | 120.5° | sp2 carbonyl angle |
| CA–C–N angle (at backbone C) | 116.6° | AMBER ff14SB ideal peptide angle |
| C–N–H angle (amide H) | 118.0° / 119.0° | sp2 amide N–H angle |
| Methyl H–C–H (tetrahedral) | 109.471° | Ideal sp3 tetrahedral angle |
| ω (both peptide bonds) | 180° (trans) | Extended / trans conformation |
| ψ (backbone) | 180° | Extended / β-sheet conformation |
| H proximity threshold (N-terminus) | 1.3 Å | Max N–H bond + margin; well below non-bonded H···N (~2.7 Å) |
| O proximity threshold (C-terminus) | 1.6 Å | Captures C–O (1.23–1.36 Å); well below non-bonded C···O (~2.8 Å) |

**Changing backbone conformation**

To change ω or ψ, modify the dihedral values in the relevant `nerf()` calls. Remember the sign convention: pass (desired standard dihedral) − 180°.

```python
# To get ω = 0° (cis peptide, rare):
ACE_C = nerf(C, CA, N,  1.335, 121.0, 180.0)  # 0° − 180° = −180° = 180°

# To get ψ = −60° (alpha-helix-like):
NME_N = nerf(N, CA, C,  1.335, 116.6, -240.0)  # −60° − 180° = −240° = 120°
```

**Adding support for a new terminus name**

If a new carboxylate oxygen name appears in a future residue, add it to `CTERM_ONAMES`:

```python
CTERM_ONAMES = {'O1', 'OXT', 'OT1', 'OT2', 'O2', 'OCT1'}  # add here
```

</details>